# Statistics for Machine Learning & Data Science
**Consolidated Laboratory Manual** | TE AI & DS | Semester V | Thadomal Shahani Engineering College

**Setup (do this once):**
1. Install libraries: `pip install pandas seaborn scikit-learn scipy matplotlib`
2. Put `iris.csv`, `titanic.csv` and `house_price.csv` in one folder, e.g. `Desktop\ML_Data\`
3. Set the folder path in the cell below (keep the `r` before the quotes and use forward slashes)

In [ ]:
%matplotlib inline
folder = r'C:/Users/YourName/Desktop/ML_Data/'

## Experiment 1: Exploratory Data Analysis (Iris)
**Aim:** Explore a dataset using descriptive statistics, histograms and a correlation matrix.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(folder + 'iris.csv')
df.columns = df.columns.str.lower()
df = df.drop(columns=[c for c in ['id'] if c in df.columns])
print(df.head())
print(df.describe())

df.hist(figsize=(8, 6))
plt.suptitle('Histograms of each feature')
plt.tight_layout()
plt.show()

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation matrix')
plt.show()

## Experiment 2: Data Preprocessing (Titanic)
**Aim:** Handle missing values, encode text columns into numbers, and scale numeric columns.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(folder + 'titanic.csv')
df.columns = df.columns.str.lower()

print('Missing values per column:')
print(df.isnull().sum())

df['age'] = df['age'].fillna(df['age'].median())    # fill missing numbers
df['fare'] = df['fare'].fillna(df['fare'].median())
df = df.dropna(subset=['embarked'])                 # drop rows missing embarked
df = pd.get_dummies(df, columns=['sex'], drop_first=True)   # text -> number
df[['age', 'fare']] = StandardScaler().fit_transform(df[['age', 'fare']])

print('\nTotal missing after cleaning:', df.isnull().sum().sum())

sns.boxplot(x=df['fare'])
plt.title('Fare after scaling (outliers show as dots)')
plt.show()

## Experiment 3: Sampling Techniques (House Price)
**Aim:** Compare simple random, stratified and systematic sampling against the full population.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(folder + 'house_price.csv')
df = df.select_dtypes(include='number').dropna()    # numeric columns only
target = df.columns[-1]       # last numeric column = the price
group_col = df.columns[0]     # any numeric column, used to make strata
n = min(500, len(df))         # sample size

simple = df.sample(n=n, random_state=1)
df['grp'] = pd.qcut(df[group_col], 4, labels=False, duplicates='drop')
strat = df.groupby('grp', group_keys=False).sample(frac=n / len(df), random_state=1)
systematic = df.iloc[::max(1, len(df) // n)]

means = {
    'Population': df[target].mean(),
    'Simple random': simple[target].mean(),
    'Stratified': strat[target].mean(),
    'Systematic': systematic[target].mean(),
}
print(means)

plt.bar(means.keys(), means.values())
plt.ylabel('Mean of ' + target)
plt.title('Sample means vs population mean')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Experiment 4: Hypothesis Testing (Titanic)
**Aim:** Run chi-square, t-test and ANOVA, and read the p-value to accept or reject the null hypothesis.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv(folder + 'titanic.csv')
df.columns = df.columns.str.lower()
df = df.dropna(subset=['age'])

# Chi-square: is survival related to sex?
chi2, p1, _, _ = stats.chi2_contingency(pd.crosstab(df['sex'], df['survived']))
print('Chi-square (sex vs survived) p =', p1)

# t-test: do survivors and non-survivors differ in age?
age_survived = df[df['survived'] == 1]['age']
age_died = df[df['survived'] == 0]['age']
t, p2 = stats.ttest_ind(age_survived, age_died)
print('t-test (age by survival) p =', p2)

# ANOVA: does fare differ across passenger classes?
groups = [g['fare'].dropna() for _, g in df.groupby('pclass')]
f, p3 = stats.f_oneway(*groups)
print('ANOVA (fare across classes) p =', p3)

sns.barplot(x='sex', y='survived', data=df)
plt.title('Survival rate by sex')
plt.show()

## Experiment 6: Linear Regression (House Price)
**Aim:** Fit a straight line to predict house price from one feature and measure the fit with R².

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

df = pd.read_csv(folder + 'house_price.csv')
df = df.select_dtypes(include='number').dropna()
target = df.columns[-1]       # the price column
feature = df.drop(columns=target).corrwith(df[target]).abs().idxmax()
print('Feature used:', feature, '| Target:', target)

X = df[[feature]]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
print('R2 score:', r2_score(y_test, pred))

order = X_test[feature].argsort()
plt.scatter(X_test, y_test, s=5, alpha=0.4, label='Actual')
plt.plot(X_test.values[order], pred[order], color='red', label='Best-fit line')
plt.xlabel(feature)
plt.ylabel(target)
plt.legend()
plt.title('Linear regression')
plt.show()

## Experiment 8: Classification Models (Iris)
**Aim:** Train Logistic Regression, KNN and Naive Bayes, compare accuracy, and view a confusion matrix.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

df = pd.read_csv(folder + 'iris.csv')
df.columns = df.columns.str.lower()
df = df.drop(columns=[c for c in ['id'] if c in df.columns])

X = df.drop(columns='species')
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, 'accuracy:', accuracy_score(y_test, m.predict(X_test)))

ConfusionMatrixDisplay.from_estimator(models['KNN'], X_test, y_test)
plt.title('KNN confusion matrix')
plt.show()

## Experiment 9: Decision Tree & Random Forest (Iris)
**Aim:** Train a single decision tree and a random forest, compare accuracy, and visualise both.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv(folder + 'iris.csv')
df.columns = df.columns.str.lower()
df = df.drop(columns=[c for c in ['id'] if c in df.columns])

X = df.drop(columns='species')
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print('Decision Tree accuracy:', accuracy_score(y_test, dt.predict(X_test)))
print('Random Forest accuracy:', accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(12, 6))
plot_tree(dt, feature_names=list(X.columns), class_names=list(dt.classes_), filled=True)
plt.title('Decision tree')
plt.show()

plt.bar(X.columns, rf.feature_importances_)
plt.title('Random forest feature importance')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()